# 03 · Voice gate: log-Mel CNN (speech vs no-voice) · fast.ai on a Kaggle T4

**Goal.** Train a tiny log-Mel classifier that decides whether the 5 s voice buffer holds a
sustained vowel at all, so traffic, fans, mixer-grinders and pressure-cooker whistles never
reach the voice-biomarker extractor. Compare the Tiny 4-block Mel CNN (target <= 0.3 MB INT8)
with `mobilenetv3_small_050`.

* Audio: the Cycle-1 voice-audit harness (`voice_synth.py`), 44.1 kHz synthesis decimated to
  the 4.41 kHz client rate, 5 s buffer. 20 speech + 9 no-voice scenarios (S0-S5, N1-N8, N6n).
* Input: `(3, 64, 101)` = log-Mel (64 bands, 50-2205 Hz, hop 220) + delta + delta-delta.
  No flips and no crops (L3): flipping time or frequency changes what the sound is.
* Target: **N4 (street traffic + horns) false accept <= 2 %** at **>= 90 % speech recall for
  SNR >= 15 dB**, on speaker-seed-disjoint test sessions.
* Kaggle settings: **GPU T4 x1**, internet on for the pinned install, and the voice harness
  dataset attached (`voice_synth.py` from the Drive voice-audit folder).

Not a medical device. Synthetic audio only.

## Shootout §6 lessons applied here

| # | Lesson (FastAI-CV-Shootout-Report §6) | Where in this notebook |
|---|---|---|
| L1 | Pre-generate the data cache once (the 9,900-session cache is ~1.3 GB float16 `.npz`) and attach it as a Kaggle dataset; never regenerate inside a GPU session | §1 Data |
| L2 | Keep the float16 tensors resident on the GPU, batch 256 with the LR scaled up, `num_workers=0` | §2 DataLoaders |
| L3 | No `vision_learner`, no `aug_transforms`, no flips or crops: the axes are time and frequency, not pixels | §2 DataLoaders, §3 Model |
| L4 | `to_fp16()` mixed precision, but keep spectral maths (DFT / power / log) in FP32 with `torch.autocast(enabled=False)` | §3 Model |
| L5 | `pretrained=False` everywhere: ImageNet weights do not transfer to spectrograms or STMaps | §3 Model |
| L6 | Pin versions (torch 2.15.0, fastai 2.8.12, timm 1.0.31, onnx 1.23.2, onnxruntime 1.29.0, Python 3.13) and export with `torch.onnx.export(dynamo=False, opset_version=17)` | §0 Setup, §5 Export |
| L7 | Budget epochs from measured speed (C3: ~2 min/epoch on CPU, < 10 s/epoch on a T4); static QDQ INT8 for convnets, never dynamic INT8 (5-30x slower on x86) | §4 Train, §5 Export |

## 0 · Setup (L6)

In [ ]:
# L6: pin versions. Kaggle images move under you; a pinned install makes reruns comparable.
!pip install -q torch==2.15.0 fastai==2.8.12 timm==1.0.31 onnx==1.23.2 onnxruntime==1.29.0

In [ ]:
import os, sys, time, json, math, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./out")
OUT.mkdir(parents=True, exist_ok=True)
print(sys.version.split()[0], torch.__version__, DEVICE,
      torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU")
# SMOKE=1 runs every cell end to end on tiny random tensors (CPU is fine); use it before a GPU run.
SMOKE = os.environ.get("SMOKE", "0") == "1"

## 1 · Voice harness and scenarios

In [ ]:
# The harness is attached as a dataset; only voice_synth.py is imported. The scenario table and
# session recipe below are vendored from voice_budget_stress.py so they match the audit exactly.
HARNESS = Path(os.environ.get("PRANASCAN_VOICE_HARNESS", "/kaggle/input/pranascan-voice-harness"))
sys.path.insert(0, str(HARNESS))
import voice_synth as vs  # noqa: E402

MASTER_SEED = 20261126
SPEECH_RMS = 0.1  # -20 dBFS vowel level at the device output (audit assumption A3)


def scenario_table():
    S = []
    sp = lambda sid, name, **kw: S.append({"id": sid, "name": name, "kind": "speech", **kw})
    nv = lambda sid, name, **kw: S.append({"id": sid, "name": name, "kind": "novoice", **kw})
    sp("S0", "Quiet room, SNR 35 dB (reference)", noise="floor", snr=35)
    sp("S1a", "Ceiling fan 50 Hz+harmonics, SNR 20 dB", noise="fan", snr=20)
    sp("S1b", "Ceiling fan, SNR 12 dB", noise="fan", snr=12)
    sp("S1c", "Ceiling fan, SNR 5 dB", noise="fan", snr=5)
    sp("S2a", "Street traffic + horns, SNR 15 dB", noise="traffic", snr=15)
    sp("S2b", "Street traffic + horns, SNR 8 dB", noise="traffic", snr=8)
    sp("S3a", "Household mix (fan+TV+pink), SNR 12 dB", noise="house", snr=12)
    sp("S3b", "Household mix, SNR 8 dB", noise="house", snr=8)
    sp("S3c", "Household mix, SNR 5 dB", noise="house", snr=5)
    sp("S4a", "Budget MEMS: HPF 150 Hz + AGC, fan SNR 20 dB", noise="fan", snr=20, mems=True, agc=True)
    sp("S4b", "Budget MEMS: AGC + clipping (+12 dB drive), quiet", noise="floor", snr=35, mems=True, agc=True, clip_db=12)
    sp("S4d", "Budget MEMS: AGC + mild clipping (+6 dB drive), quiet", noise="floor", snr=35, mems=True, agc=True, clip_db=6)
    sp("S4c", "Budget MEMS: AGC + clipping + fan SNR 12 dB", noise="fan", snr=12, mems=True, agc=True, clip_db=12)
    for d in (0.5, 1.0, 1.5, 2.0, 2.5):
        sp(f"S5-{d}", f"Short vowel {d:.1f} s in 5 s buffer, quiet", noise="floor", snr=35, dur=d)
    sp("S5-1.5fan", "Short vowel 1.5 s, ceiling fan SNR 12 dB", noise="fan", snr=12, dur=1.5)
    sp("S5-buf1.9", "Whole buffer only 1.9 s (early stop)", noise="floor", snr=35, buffer_s=1.9, dur=1.4)
    nv("N1", "Digital silence", noise="zero", level_db=-200)
    nv("N2", "Quiet room only (-55 dBFS)", noise="floor", level_db=-55)
    nv("N3", "Ceiling fan only (-32 dBFS, = SNR 12 dB)", noise="fan", level_db=-32)
    nv("N4", "Street traffic + horns only (-30 dBFS)", noise="traffic", level_db=-30)
    nv("N5", "Mixer-grinder only (-26 dBFS)", noise="mixer", level_db=-26)
    nv("N6", "Pressure-cooker whistle (-26 dBFS), anti-aliased decimation", noise="cooker", level_db=-26)
    nv("N6n", "Pressure-cooker whistle (-26 dBFS), naive decimation", noise="cooker", level_db=-26, naive=True)
    nv("N7", "Quiet fan (-50 dBFS) through budget AGC", noise="fan", level_db=-50, mems=True, agc=True)
    nv("N8", "Household mix only (-32 dBFS)", noise="house", level_db=-32)
    return S


NOISE = {"floor": vs.room_floor, "fan": vs.ceiling_fan, "traffic": vs.street_traffic,
         "house": vs.household_mix, "mixer": vs.mixer_grinder, "cooker": vs.pressure_cooker_whistle}


def draw_subject(rng):
    male = rng.random() < 0.5
    return {"f0": rng.uniform(85, 150) if male else rng.uniform(165, 280),
            "formant_scale": rng.uniform(0.95, 1.05) if male else rng.uniform(1.10, 1.20),
            "jit": rng.uniform(0.3, 1.5), "shim": rng.uniform(2.0, 7.0),
            "onset": rng.uniform(0.3, 0.7), "tail": rng.uniform(0.2, 0.5)}


def make_session(sc, rng):
    buf = sc.get("buffer_s", vs.BUFFER_S)
    n = int(round(buf * vs.FS_HI))
    if sc["kind"] == "speech":
        subj = draw_subject(rng)
        dur = sc.get("dur", buf - subj["onset"] - subj["tail"])
        onset = subj["onset"] if dur + subj["onset"] < buf - 0.05 else 0.1
        v, truth = vs.sustained_vowel(rng, subj["f0"], dur, onset, subj["jit"], subj["shim"],
                                      subj["formant_scale"], buffer_s=buf)
        a, b = truth.onset_s, truth.onset_s + truth.duration_s
        v *= SPEECH_RMS / np.sqrt(np.mean(v[int(a * vs.FS_HI): int(b * vs.FS_HI)] ** 2))
        noise = NOISE[sc["noise"]](rng, n)
        floor = vs.room_floor(rng, n)
        ps, pn = vs.band_power(v, a, b), vs.band_power(noise, a, b)
        noise *= np.sqrt(ps / pn / 10 ** (sc["snr"] / 10))
        floor *= np.sqrt(ps / vs.band_power(floor, a, b) / 10 ** 3.5)
        x = v + noise + (floor if sc["noise"] != "floor" else 0)
    else:
        x = np.zeros(n) if sc["noise"] == "zero" else NOISE[sc["noise"]](rng, n)
        if sc["noise"] != "zero":
            x *= 10 ** (sc["level_db"] / 20) / np.std(x)
    if sc.get("mems"):
        x = vs.mems_highpass(x)
    if sc.get("agc"):
        x = vs.agc(x)
    if sc.get("clip_db"):
        x, _ = vs.hard_clip(x, sc["clip_db"])
    return vs.to_client_rate(x, naive=sc.get("naive", False))


SCENARIOS = scenario_table()
print(sum(s["kind"] == "speech" for s in SCENARIOS), "speech /",
      sum(s["kind"] == "novoice" for s in SCENARIOS), "no-voice scenarios")

## 1b · Log-Mel + delta + delta-delta front end (L3)

In [ ]:
FS_LO, BUF_S, N_FFT, HOP, N_MELS = 4410, 5.0, 512, 220, 64
N_LO = int(FS_LO * BUF_S)  # 22050 samples -> 1 + 22050 // 220 = 101 frames


def mel_filterbank(sr=FS_LO, n_fft=N_FFT, n_mels=N_MELS, fmin=50.0, fmax=None):
    fmax = fmax or sr / 2
    hz2mel = lambda f: 2595 * np.log10(1 + f / 700.0)
    mel2hz = lambda m: 700 * (10 ** (m / 2595) - 1)
    pts = mel2hz(np.linspace(hz2mel(fmin), hz2mel(fmax), n_mels + 2))
    bins = np.fft.rfftfreq(n_fft, 1 / sr)
    fb = np.zeros((n_mels, bins.size))
    for i in range(n_mels):
        lo, c, hi = pts[i], pts[i + 1], pts[i + 2]
        fb[i] = np.clip(np.minimum((bins - lo) / (c - lo), (hi - bins) / (hi - c)), 0, None)
    return fb


MEL_FB = mel_filterbank()
WIN = np.hanning(N_FFT + 1)[:-1]


def delta(x, w=2):
    pad = np.pad(x, ((0, 0), (w, w)), mode="edge")
    num = sum(k * (pad[:, w + k: w + k + x.shape[1]] - pad[:, w - k: w - k + x.shape[1]])
              for k in range(1, w + 1))
    return num / (2 * sum(k * k for k in range(1, w + 1)))


def logmel3(x_lo):
    """4.41 kHz buffer -> (3, 64, 101) float16: log-Mel, delta, delta-delta. No per-clip
    normalisation: absolute level is part of what separates silence from a vowel."""
    x = np.zeros(N_LO); x[: min(N_LO, len(x_lo))] = x_lo[:N_LO]   # short buffers are zero-padded
    xp = np.pad(x, N_FFT // 2, mode="reflect")
    frames = np.lib.stride_tricks.sliding_window_view(xp, N_FFT)[::HOP][: 1 + N_LO // HOP]
    pw = np.abs(np.fft.rfft(frames * WIN, axis=1)) ** 2
    lm = np.log(MEL_FB @ pw.T + 1e-10).astype(np.float32)           # (64, 101)
    d1 = delta(lm); d2 = delta(d1)
    return np.stack([lm, d1, d2]).astype(np.float16)


print(logmel3(np.zeros(N_LO)).shape)

## 1c · Build or load the feature cache (L1)

In [ ]:
# L1: build the feature cache once (~2-4 min on Kaggle CPU), then save /kaggle/working/voice_cache
# as a dataset and attach it next time. Splits are speaker-seed-disjoint: each split has its own
# root seed, and every session draws a fresh speaker (f0, formants, jitter, shimmer) from it.
SPLIT_SEEDS = {"train": MASTER_SEED + 1, "val": MASTER_SEED + 2, "test": MASTER_SEED + 3}
N_PER = {"train": 120, "val": 40, "test": 60}  # sessions per scenario
if SMOKE:
    N_PER = {"train": 2, "val": 2, "test": 2}
CACHED = Path(os.environ.get("PRANASCAN_VOICE_CACHE", "/kaggle/input/pranascan-voice-cache"))


def build_split(split):
    path = CACHED / f"voice_{split}_{N_PER[split]}.npz"
    if path.exists():
        with np.load(path, allow_pickle=False) as z:
            return {k: z[k] for k in z.files}
    root = np.random.SeedSequence(SPLIT_SEEDS[split])
    xs, ys, sids, snrs = [], [], [], []
    for sc, child in zip(SCENARIOS, root.spawn(len(SCENARIOS))):
        rng = np.random.default_rng(child)
        for _ in range(N_PER[split]):
            xs.append(logmel3(make_session(sc, rng)))
            ys.append(int(sc["kind"] == "speech")); sids.append(sc["id"]); snrs.append(sc.get("snr", -1))
    d = {"x": np.stack(xs), "y": np.array(ys, np.int64), "scenario": np.array(sids),
         "snr": np.array(snrs, np.float32)}
    np.savez(OUT / f"voice_{split}_{N_PER[split]}.npz", **d)
    return d


t0 = time.time()
tr, va, te = (build_split(s) for s in ("train", "val", "test"))
print({k: d["x"].shape for k, d in (("train", tr), ("val", va), ("test", te))},
      f"{time.time() - t0:.0f} s")
MU, SD = tr["x"].astype(np.float32).mean((0, 2, 3), keepdims=True), tr["x"].astype(np.float32).std((0, 2, 3), keepdims=True) + 1e-6
norm = lambda x: ((x.astype(np.float32) - MU) / SD).astype(np.float16)  # one global, per-channel scaler

## 2 · DataLoaders (L2, L3)

In [ ]:
def gpu_dls(xtr, ytr, xva, yva, bs=256):
    """L2: float16 tensors resident on the GPU, no worker processes, no batch transforms (L3)."""
    from fastai.data.core import DataLoaders
    from torch.utils.data import TensorDataset

    to = lambda a, dt: torch.as_tensor(a).to(DEVICE, dt)
    ds_tr = TensorDataset(to(xtr, torch.float16), to(ytr, ytr_dtype(ytr)))
    ds_va = TensorDataset(to(xva, torch.float16), to(yva, ytr_dtype(yva)))
    return DataLoaders.from_dsets(ds_tr, ds_va, bs=bs, num_workers=0, pin_memory=False,
                                  device=DEVICE)


def ytr_dtype(y):
    return torch.long if np.asarray(y).dtype.kind in "iu" else torch.float32


class Fp32Input(nn.Module):
    """Cast the float16 batch up once; to_fp16 then autocasts the convs back to half."""

    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        return self.m(x.float())


# L2: the reference recipe used bs=64, lr=3e-3; at bs=256 scale the LR by sqrt(256/64) = 2.
BS, LR = 256, 3e-3 * math.sqrt(256 / 64)

## 3 · Models (L5)

In [ ]:
class TinyMelCNN(nn.Module):
    """4 x (conv-BN-ReLU-maxpool) on (3, 64, 101); ~25k params, well under 0.3 MB as INT8."""

    def __init__(self, n_classes=2, widths=(16, 24, 32, 48)):
        super().__init__()
        layers, c = [], 3
        for w in widths:
            layers += [nn.Conv2d(c, w, 3, padding=1, bias=False), nn.BatchNorm2d(w),
                       nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            c = w
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(c, n_classes)

    def forward(self, x):
        return self.head(self.features(x).mean((2, 3)))


def build(arch):
    if arch == "tiny_mel_cnn":
        return TinyMelCNN()
    if arch == "mobilenetv3_small_050":
        import timm
        return timm.create_model("mobilenetv3_small_050", pretrained=False, in_chans=3,
                                 num_classes=2)  # L5
    raise KeyError(arch)


ARCHS = ["tiny_mel_cnn", "mobilenetv3_small_050"]
for a in ARCHS:
    print(a, sum(p.numel() for p in build(a).parameters()), "params")

## 4 · Train both (L4, L7)

In [ ]:
from fastai.callback.schedule import fit_one_cycle  # noqa: F401
from fastai.learner import Learner
from fastai.losses import CrossEntropyLossFlat
from fastai.metrics import accuracy

counts = np.bincount(tr["y"], minlength=2)
weight = torch.tensor(counts.sum() / (2 * np.maximum(counts, 1)), dtype=torch.float32).to(DEVICE)
dls = gpu_dls(norm(tr["x"]), tr["y"], norm(va["x"]), va["y"], bs=BS)
nets = {}
for arch in ARCHS:
    torch.manual_seed(SEED)
    learn = Learner(dls, Fp32Input(build(arch)), loss_func=CrossEntropyLossFlat(weight=weight),
                    metrics=accuracy)
    if DEVICE.type == "cuda":
        learn = learn.to_fp16()
    t0 = time.time()
    with learn.no_bar():
        learn.fit_one_cycle(2 if SMOKE else 20, LR)
    nets[arch] = learn.model.m.float().eval()
    torch.save(nets[arch].state_dict(), OUT / f"voice_{arch}.pt")
    print(arch, f"{time.time() - t0:.0f} s")

## 4b · Operating point on val, N4 false accept on test

In [ ]:
def p_speech(m, x, bs=1024):
    m = m.to(DEVICE).eval(); out = []
    with torch.no_grad():
        for i in range(0, len(x), bs):
            xb = torch.as_tensor(norm(x[i:i + bs])).to(DEVICE).float()
            out.append(torch.softmax(m(xb), 1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def gate_report(p, d, thr):
    hi = (d["y"] == 1) & (d["snr"] >= 15)
    out = {"thr": round(float(thr), 4),
           "speech_recall_snr15": round(float((p[hi] >= thr).mean()), 3),
           "speech_recall_all": round(float((p[d["y"] == 1] >= thr).mean()), 3)}
    for sid in ("N1", "N2", "N3", "N4", "N5", "N6", "N6n", "N7", "N8"):
        m = d["scenario"] == sid
        if m.any():
            out[f"FA_{sid}"] = round(float((p[m] >= thr).mean()), 3)
    out["meets_target"] = out["speech_recall_snr15"] >= 0.90 and out.get("FA_N4", 1.0) <= 0.02
    return out


rows = {}
for arch, m in nets.items():
    pv, pt = p_speech(m, va["x"]), p_speech(m, te["x"])
    hi = (va["y"] == 1) & (va["snr"] >= 15)
    thr = float(np.quantile(pv[hi], 0.10))  # val: the largest cut that keeps >= 90 % recall @ SNR >= 15
    rows[arch] = gate_report(pt, te, thr)
    print(arch, rows[arch])
json.dump(rows, open(OUT / "voice_gate_test.json", "w"), indent=2)

## 5 · Export: FP32 + static INT8 QDQ ONNX (L6, L7)

In [ ]:
def export_onnx(model, shape, path):
    """L6: TorchScript exporter (dynamo=False), opset 17, dynamic batch axis."""
    model = model.float().eval().cpu()
    torch.onnx.export(model, (torch.randn(*shape),), str(path), dynamo=False, opset_version=17,
                      input_names=["x"], output_names=["logits"],
                      dynamic_axes={"x": {0: "batch"}, "logits": {0: "batch"}})
    return path


def quantize_qdq(fp32_path, out_path, calib, op_types=None):
    """L7: static INT8 QDQ (per-channel weights). Dynamic INT8 is 5-30x slower on x86 CPUs."""
    from onnxruntime.quantization import (CalibrationDataReader, QuantFormat, QuantType,
                                          quantize_static)
    from onnxruntime.quantization.shape_inference import quant_pre_process

    class Reader(CalibrationDataReader):
        def __init__(self, x):
            self.it = iter([{"x": x[i:i + 1].astype(np.float32)} for i in range(len(x))])

        def get_next(self):
            return next(self.it, None)

    pre = str(out_path).replace(".onnx", "_pre.onnx")
    quant_pre_process(str(fp32_path), pre, skip_symbolic_shape=True)
    kw = dict(quant_format=QuantFormat.QDQ, per_channel=True,
              activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
    if op_types:
        kw["op_types_to_quantize"] = op_types
    quantize_static(pre, str(out_path), Reader(calib), **kw)
    os.remove(pre)
    return out_path


def ort_logits(path, x, bs=128):
    import onnxruntime as ort
    so = ort.SessionOptions(); so.intra_op_num_threads = 4
    s = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
    return np.concatenate([s.run(None, {"x": x[i:i + bs].astype(np.float32)})[0]
                           for i in range(0, len(x), bs)])


def cpu1_latency_ms(path, shape, warm=30, runs=200):
    import onnxruntime as ort
    so = ort.SessionOptions(); so.intra_op_num_threads = 1; so.inter_op_num_threads = 1
    s = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
    x = np.random.default_rng(0).standard_normal(shape).astype(np.float32)
    for _ in range(warm):
        s.run(None, {"x": x})
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter(); s.run(None, {"x": x}); ts.append((time.perf_counter() - t0) * 1e3)
    return float(np.median(ts))

In [ ]:
SHAPE = (1, 3, 64, 101)
calib = norm(va["x"][:: max(1, len(va["x"]) // 256)]).astype(np.float32)
for arch, m in nets.items():
    fp32 = export_onnx(m, SHAPE, OUT / f"voice_{arch}_fp32.onnx")
    int8 = quantize_qdq(fp32, OUT / f"voice_{arch}_int8_static.onnx", calib)
    for p in (fp32, int8):
        lo = ort_logits(p, norm(te["x"]).astype(np.float32))
        ps = np.exp(lo[:, 1]) / np.exp(lo).sum(1)
        print(p.name, f"{os.path.getsize(p) / 1e6:.3f} MB", f"cpu1 {cpu1_latency_ms(p, SHAPE):.2f} ms",
              gate_report(ps, te, rows[arch]["thr"]))
np.savez(OUT / "voice_feature_scaler.npz", mu=MU, sd=SD)  # the phone must apply the same scaler

## Next

* Ship the smaller model that meets the target, with `voice_feature_scaler.npz` and its `thr`.
* The gate only decides "is there a vowel". It never scores voice biomarkers.
* N6n (naive decimation) is a client bug check: if it passes the gate, fix the decimator first.